<a href="https://colab.research.google.com/github/devslayer69/dummy/blob/main/repaso_p2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
%pip install pandas numpy ydata-profiling scikit-learn statsmodels

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import pandas as pd
import numpy as np
from ydata_profiling import ProfileReport

df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/datos/creditos.csv')

ProfileReport(df)

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 7/7 [00:00<00:00, 103.61it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

monto_credito
ingreso_mensual
score_crediticio
deuda_actual
tasa_interes

In [7]:
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
x = df.drop(columns='monto_credito')
y = df['monto_credito']

x_train, x_test, y_train, y_test = train_test_split(
    x, y, train_size=0.80, random_state=42)

for col in ['ingreso_mensual', 'score_crediticio', 'deuda_actual', 'tasa_interes']:
  x_train[col] = x_train[col].clip(upper=x_train[col].quantile(0.99), lower=x_train[col].quantile(0.01))

y_train = y_train.clip(upper=y_train.quantile(0.99), lower=y_train.quantile(0.01))

In [8]:
import statsmodels.api as sm
x_trainc = sm.add_constant(x_train)
rlm = sm.RLM(y_train, x_trainc, M=sm.robust.norms.HuberT())
rlm.fit().summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                    Robust Linear Model Regression Results                    
==============================================================================
Dep. Variable:          monto_credito   No. Observations:                  320
Model:                            RLM   Df Residuals:                      313
Method:                          IRLS   Df Model:                            6
Norm:                          HuberT                                         
Scale Est.:                       mad                                         
Cov Type:                          H1                                         
Date:                Tue, 20 Oct 2026                                         
Time:                        04:12:27                                         
No. Iterations:                     7                                         
======================================================================================
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
const              -5.867e+04   1.04e+04     -5.623      0.000   -7.91e+04   -3.82e+04
ingreso_mensual        2.8802      0.098     29.457      0.000       2.689       3.072
score_crediticio      93.8901      9.260     10.139      0.000      75.741     112.040
deuda_actual          -0.5958      0.035    -16.798      0.000      -0.665      -0.526
edad                  86.1262     30.935      2.784      0.005      25.494     146.758
antiguedad_laboral   240.7774     58.490      4.117      0.000     126.140     355.415
tasa_interes        -200.9638    239.000     -0.841      0.400    -669.396     267.468
======================================================================================

If the model instance has been used for another fit with different fit parameters, then the fit options might not be the correct ones anymore .
"""

In [9]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
y_pred = rlm.fit().predict(sm.add_constant(x_train))
print(f'R2: {r2_score(y_train, y_pred)}\n',
      f'mae: {mean_absolute_error(y_train, y_pred)}\n',
      f'mse: {mean_squared_error(y_train, y_pred)}\n',
      f'rmse: {np.sqrt(mean_squared_error(y_train, y_pred))}')


R2: 0.8239475511467338
 mae: 4034.665675032932
 mse: 30157388.31774962
 rmse: 5491.574302306181


In [10]:
from sklearn.tree import DecisionTreeRegressor
dummy = DecisionTreeRegressor(random_state=42)
dummy.fit(x_train, y_train)
y_pred = dummy.predict(x_test)

print(np.sqrt(mean_squared_error(y_test, y_pred)), '\nmae',
mean_absolute_error(y_test, y_pred))

9434.403279728731 
mae 6719.705875


In [11]:
from sklearn.ensemble import RandomForestRegressor
dummy = RandomForestRegressor(random_state=42, n_estimators=100)
dummy.fit(x_train, y_train)

y_pred = dummy.predict(x_test)

print(np.sqrt(mean_squared_error(y_test, y_pred)), '\nmae',
mean_absolute_error(y_test, y_pred))

4726.8668316321855 
mae 3638.429061475001


In [12]:
importancias = pd.Series(dummy.feature_importances_, index=x_train.columns)
importancias.sort_values(ascending=False)

,0
score_crediticio,0.399198
ingreso_mensual,0.373907
deuda_actual,0.080531
tasa_interes,0.063974
edad,0.045634
antiguedad_laboral,0.036756
